# Feature Engineering v2: Aggregate First, Join Contact Last

This notebook rewrites the original feature engineering pipeline to fix issues found during EDA:

1. Avoid early data loss by aggregating activity first and joining contact metadata only after developer-level features are built.
2. Handle zero-inflated behavior with binary flags plus `LOG1P` features.
3. Replace nested cumulative windows with non-overlapping recency windows: `0_30d`, `30_90d`, and `90_180d`.
4. Add recency, velocity, rate, and interaction features.
5. Normalize persona lane scores and add entropy so mixed personas are represented more honestly.
6. Add feature validation checks after major table creation.

Recommended final output: `dev_profile_final_v4`.

In [54]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = "developer_project.duckdb"
con = duckdb.connect(DB_PATH)

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)

ACTIVITY_TABLE = "activity_final"
CONTACT_TABLE = "contact_final"

# Robust helper: use natural log of 1+x in DuckDB.
LOG1P = "LN(1 + {x})"

## 1. Confirm source tables and columns

In [55]:
tables = con.execute("SHOW TABLES").fetchdf()
display(tables)

available = set(tables.iloc[:, 0].astype(str))
missing = {ACTIVITY_TABLE, CONTACT_TABLE} - available
if missing:
    raise ValueError(f"Missing required table(s): {missing}")

def get_columns(table_name: str) -> set:
    return set(con.execute(f"DESCRIBE {table_name}").fetchdf()["column_name"].astype(str))

activity_cols = get_columns(ACTIVITY_TABLE)
contact_cols = get_columns(CONTACT_TABLE)

required_activity_cols = {"dev_contact", "activity_date", "activity"}
required_contact_cols = {"developer_id"}

if required_activity_cols - activity_cols:
    raise ValueError(f"Missing activity columns: {required_activity_cols - activity_cols}")
if required_contact_cols - contact_cols:
    raise ValueError(f"Missing contact columns: {required_contact_cols - contact_cols}")

for table in [ACTIVITY_TABLE, CONTACT_TABLE]:
    print(f"\n{table}")
    display(con.execute(f"SELECT COUNT(*) AS rows FROM {table}").fetchdf())
    display(con.execute(f"DESCRIBE {table}").fetchdf())

,name
0,activity_base_v2
1,activity_final
2,activity_ontology_v2
3,activity_raw
4,contact_final
5,contact_one_row_v2
6,contact_raw
7,contact_supplement_raw
8,dev_activation_v2
9,dev_contact_persona_v2



activity_final


,rows
0,69347501


,column_name,column_type,null,key,default,extra
0,dev_contact,VARCHAR,YES,None,None,None
1,activity,VARCHAR,YES,None,None,None
2,activity_name,VARCHAR,YES,None,None,None
3,activity_type,VARCHAR,YES,None,None,None
4,activity_role,VARCHAR,YES,None,None,None
5,activity_attendance,VARCHAR,YES,None,None,None
6,activity_score,DOUBLE,YES,None,None,None
7,activity_date,DATE,YES,None,None,None
8,activity_id,VARCHAR,YES,None,None,None
9,filepath,VARCHAR,YES,None,None,None



contact_final


,rows
0,9381490


,column_name,column_type,null,key,default,extra
0,developer_id,VARCHAR,YES,None,None,None
1,program_application_source,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,sub_region,VARCHAR,YES,None,None,None
5,zone,VARCHAR,YES,None,None,None
6,territory,VARCHAR,YES,None,None,None
7,organization_english_name,VARCHAR,YES,None,None,None
8,development_areas,VARCHAR,YES,None,None,None
9,other_development_areas,VARCHAR,YES,None,None,None


## 2. Source sanity checks

In [56]:
dup_name_col = "activity_name" if "activity_name" in activity_cols else "activity"
source_validation = con.execute(f"""
WITH activity_checks AS (
    SELECT
        COUNT(*) AS activity_rows,
        COUNT(DISTINCT CAST(dev_contact AS VARCHAR)) AS activity_developers,
        SUM(CASE WHEN dev_contact IS NULL OR TRIM(CAST(dev_contact AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_dev_contact,
        SUM(CASE WHEN activity_date IS NULL THEN 1 ELSE 0 END) AS missing_activity_date,
        SUM(CASE WHEN activity_date > CURRENT_DATE THEN 1 ELSE 0 END) AS future_activity_dates,
        COUNT(*) - COUNT(DISTINCT (
            COALESCE(CAST(dev_contact AS VARCHAR), '') || '|' ||
            COALESCE(CAST(activity_date AS VARCHAR), '') || '|' ||
            COALESCE(CAST(activity AS VARCHAR), '') || '|' ||
            COALESCE(CAST({dup_name_col} AS VARCHAR), '')
        )) AS possible_duplicate_activity_rows
    FROM {ACTIVITY_TABLE}
),
contact_checks AS (
    SELECT
        COUNT(*) AS contact_rows,
        COUNT(DISTINCT CAST(developer_id AS VARCHAR)) AS contact_developers,
        SUM(CASE WHEN developer_id IS NULL OR TRIM(CAST(developer_id AS VARCHAR)) = '' THEN 1 ELSE 0 END) AS missing_developer_id,
        COUNT(*) - COUNT(DISTINCT CAST(developer_id AS VARCHAR)) AS duplicate_contact_rows
    FROM {CONTACT_TABLE}
)
SELECT * FROM activity_checks CROSS JOIN contact_checks
""").fetchdf()

display(source_validation)

100% ▕██████████████████████████████████████▏ (00:00:03.50 elapsed)     


,activity_rows,activity_developers,missing_dev_contact,missing_activity_date,future_activity_dates,possible_duplicate_activity_rows,contact_rows,contact_developers,missing_developer_id,duplicate_contact_rows
0,69347501,7660278,0.0,0.0,0.0,33695697,9381490,9381490,0.0,0


## 3. Build an activity-only base table

Important design change: this table does **not** join contact metadata. Activity features are created from activity rows first so unmatched contacts do not cause early population loss and duplicate contact rows do not multiply activity rows.

In [57]:
def activity_col_expr(col: str, out: str = None, cast: str = "VARCHAR", default: str = "NULL") -> str:
    out = out or col
    if col in activity_cols:
        return f"CAST(a.{col} AS {cast}) AS {out}"
    return f"CAST({default} AS {cast}) AS {out}"

activity_score_expr = (
    "LEAST(GREATEST(COALESCE(TRY_CAST(a.activity_score AS DOUBLE), 0.0), 0.0), 100.0) AS activity_score"
    if "activity_score" in activity_cols else
    "CAST(0.0 AS DOUBLE) AS activity_score"
)

con.execute(f"""
CREATE OR REPLACE TABLE activity_base_v2 AS
SELECT
    CAST(a.dev_contact AS VARCHAR) AS developer_id,
    CAST(a.activity_date AS DATE) AS activity_date,
    LOWER(TRIM(CAST(a.activity AS VARCHAR))) AS activity,
    {activity_col_expr('activity_name')},
    {activity_col_expr('activity_type')},
    {activity_col_expr('activity_role')},
    {activity_col_expr('activity_attendance')},
    {activity_score_expr},
    {activity_col_expr('filepath')},
    {activity_col_expr('lead_source')},
    {activity_col_expr('lead_source_details')},
    {activity_col_expr('nvidia_campaign_id')}
FROM {ACTIVITY_TABLE} a
WHERE a.dev_contact IS NOT NULL
  AND TRIM(CAST(a.dev_contact AS VARCHAR)) <> ''
  AND a.activity_date IS NOT NULL
""")

display(con.execute("""
SELECT
    (SELECT COUNT(*) FROM activity_final WHERE dev_contact IS NOT NULL AND TRIM(CAST(dev_contact AS VARCHAR)) <> '' AND activity_date IS NOT NULL) AS valid_source_activity_rows,
    COUNT(*) AS activity_base_rows,
    COUNT(DISTINCT developer_id) AS activity_base_developers,
    MIN(activity_date) AS min_activity_date,
    MAX(activity_date) AS max_activity_date,
    MIN(activity_score) AS min_activity_score,
    MAX(activity_score) AS max_activity_score
FROM activity_base_v2
""").fetchdf())

100% ▕██████████████████████████████████████▏ (00:00:07.25 elapsed)     


,valid_source_activity_rows,activity_base_rows,activity_base_developers,min_activity_date,max_activity_date,min_activity_score,max_activity_score
0,69347501,69347501,7660278,2020-01-01,2026-03-12,0.0,100.0


## 4. Deduplicate contact metadata separately

This table is used only for final enrichment and profile-text persona hints. It is not joined into activity rows.

In [58]:
def contact_select_expr(col: str, out: str = None, cast: str = "VARCHAR", default: str = "NULL") -> str:
    out = out or col
    if col in contact_cols:
        return f"CAST({col} AS {cast}) AS {out}"
    return f"CAST({default} AS {cast}) AS {out}"

order_terms = []
if "last_modified_date" in contact_cols:
    order_terms.append("last_modified_date DESC NULLS LAST")
if "created_date" in contact_cols:
    order_terms.append("created_date DESC NULLS LAST")
order_by = ", ".join(order_terms) if order_terms else "developer_id"

contact_fields = [
    "developer_id", "created_date", "first_activity_date", "last_activity_date",
    "development_areas", "fields_of_interest", "account_id", "account_type",
    "country", "region", "industry_segment_vertical", "program_application_source",
    "organization_english_name", "normalized_account_name", "wwfo_category", "wwfo_target_list"
]

select_list = []
for col in contact_fields:
    if col == "developer_id":
        select_list.append("CAST(developer_id AS VARCHAR) AS developer_id")
    elif col in {"created_date", "first_activity_date", "last_activity_date"} and col in contact_cols:
        select_list.append(f"CAST({col} AS DATE) AS {col}")
    else:
        select_list.append(contact_select_expr(col))

con.execute(f"""
CREATE OR REPLACE TABLE contact_one_row_v2 AS
WITH ranked AS (
    SELECT
        {', '.join(select_list)},
        ROW_NUMBER() OVER (PARTITION BY CAST(developer_id AS VARCHAR) ORDER BY {order_by}) AS rn
    FROM {CONTACT_TABLE}
    WHERE developer_id IS NOT NULL
      AND TRIM(CAST(developer_id AS VARCHAR)) <> ''
)
SELECT * EXCLUDE (rn)
FROM ranked
WHERE rn = 1
""")

display(con.execute("""
SELECT
    COUNT(*) AS contact_one_row_rows,
    COUNT(DISTINCT developer_id) AS contact_one_row_developers
FROM contact_one_row_v2
""").fetchdf())

100% ▕██████████████████████████████████████▏ (00:00:04.10 elapsed)     


,contact_one_row_rows,contact_one_row_developers
0,9381490,9381490


## 5. Build activity ontology without contact join

In [59]:
con.execute(r"""
CREATE OR REPLACE TABLE activity_ontology_v2 AS
WITH base AS (
    SELECT
        *,
        LOWER(
            COALESCE(activity, '') || ' ' ||
            COALESCE(activity_type, '') || ' ' ||
            COALESCE(activity_role, '') || ' ' ||
            COALESCE(activity_attendance, '') || ' ' ||
            COALESCE(lead_source, '')
        ) AS keyword_text,
        LOWER(
            COALESCE(activity_name, '') || ' ' ||
            COALESCE(filepath, '') || ' ' ||
            COALESCE(lead_source_details, '')
        ) AS persona_activity_text
    FROM activity_base_v2
),
scored AS (
    SELECT
        *,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'cuda|cudnn|rapids|nccl|cutlass|\bdali\b|accelerated[ -]?computing|nsight|optix|dcgm|gpu operator|mig|hpc|nvapi|video codec sdk') THEN 3.0 ELSE 0.0 END AS cuda_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'triton|tensorrt|nemo|\bnim\b|llm|large language model|generative ai|genai|rag|retrieval augmented|inference microservice|ai enterprise|deep learning|machine learning|pytorch|tensorflow|onnx') THEN 3.0 ELSE 0.0 END AS genai_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'isaac|robot|ros|autonomous machine|jetson|edge ai|embedded ai|metropolis') THEN 3.0 ELSE 0.0 END AS robotics_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'omniverse|simulation|digital twin|modulus|physics|render|rtx|openusd|usd|graphics') THEN 3.0 ELSE 0.0 END AS simulation_activity_score,
        CASE WHEN REGEXP_MATCHES(persona_activity_text, 'dli|training|course|workshop|webinar|certification|academy|learn|bootcamp|community|forum|developer program|gputechconf|gtc') THEN 2.0 ELSE 0.0 END AS learning_community_activity_score
    FROM base
)
SELECT
    *,
    CASE
        WHEN REGEXP_MATCHES(keyword_text, 'champion|ambassador|speaker|presenter|instructor|mentor|contributor|advocate') THEN 'Champion'
        WHEN REGEXP_MATCHES(keyword_text, 'build|deploy|api|sdk|container|workspace|notebook|project|sample|github|repo|download') THEN 'Build'
        WHEN REGEXP_MATCHES(keyword_text, 'trial|eval|benchmark|test|demo|proof|poc|assessment') THEN 'Evaluate'
        WHEN REGEXP_MATCHES(keyword_text, 'learn|training|course|webinar|workshop|certification|dli|tutorial') THEN 'Learn'
        WHEN REGEXP_MATCHES(keyword_text, 'discover|awareness|newsletter|event|campaign|page view|visit|content') THEN 'Discover'
        ELSE 'Discover'
    END AS journey_signal,
    CASE
        WHEN REGEXP_MATCHES(keyword_text, 'deploy|build|api|sdk|container|workspace|notebook|project|github|repo|champion|speaker|contributor') THEN 'High'
        WHEN REGEXP_MATCHES(keyword_text, 'trial|eval|benchmark|test|demo|download|course|workshop|training') THEN 'Moderate'
        ELSE 'Passive'
    END AS effort_level,
    CASE
        WHEN GREATEST(cuda_activity_score, genai_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) = 0 THEN 'Other'
        WHEN cuda_activity_score >= GREATEST(genai_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'CUDA'
        WHEN genai_activity_score >= GREATEST(cuda_activity_score, robotics_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'GenAI'
        WHEN robotics_activity_score >= GREATEST(cuda_activity_score, genai_activity_score, simulation_activity_score, learning_community_activity_score) THEN 'Robotics'
        WHEN simulation_activity_score >= GREATEST(cuda_activity_score, genai_activity_score, robotics_activity_score, learning_community_activity_score) THEN 'Simulation'
        ELSE 'Learning_Community'
    END AS persona_hint,
    cuda_activity_score AS cuda_persona_score,
    genai_activity_score AS genai_persona_score,
    robotics_activity_score AS robotics_persona_score,
    simulation_activity_score AS simulation_persona_score,
    learning_community_activity_score AS learning_community_persona_score,
    CASE
        WHEN REGEXP_MATCHES(keyword_text, 'download') THEN 'Download'
        WHEN REGEXP_MATCHES(keyword_text, 'api|hosted|endpoint|nim') THEN 'Hosted API'
        WHEN REGEXP_MATCHES(keyword_text, 'workspace|notebook|lab') THEN 'Cloud Workspace'
        WHEN REGEXP_MATCHES(keyword_text, 'forum|community|ambassador|champion') THEN 'Community'
        WHEN REGEXP_MATCHES(keyword_text, 'training|course|dli|certification|workshop') THEN 'Training'
        WHEN REGEXP_MATCHES(keyword_text, 'event|webinar|conference|gtc') THEN 'Event'
        ELSE 'Content'
    END AS modality
FROM scored
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    SUM(CASE WHEN persona_hint <> 'Other' THEN 1 ELSE 0 END) AS persona_mapped_rows
FROM activity_ontology_v2
""").fetchdf())

display(con.execute("""
SELECT journey_signal, effort_level, persona_hint, modality, COUNT(*) AS rows
FROM activity_ontology_v2
GROUP BY 1,2,3,4
ORDER BY rows DESC
LIMIT 30
""").fetchdf())

100% ▕██████████████████████████████████████▏ (00:00:24.87 elapsed)     


,rows,developers,persona_mapped_rows
0,69347501,7660278,46056702.0


,journey_signal,effort_level,persona_hint,modality,rows
0,Build,Moderate,CUDA,Download,23011832
1,Build,Moderate,Robotics,Download,9940737
2,Build,Moderate,Other,Download,8580974
3,Build,High,Other,Hosted API,6146114
4,Discover,Passive,Other,Content,6109819
5,Build,High,GenAI,Download,2473397
6,Discover,Passive,CUDA,Content,2469613
7,Build,High,Other,Download,1415199
8,Build,High,CUDA,Download,1169475
9,Discover,Passive,Learning_Community,Content,933645


## 6. Developer universe and anchor date

In [60]:
con.execute(f"""
CREATE OR REPLACE TABLE developer_universe_v2 AS
SELECT DISTINCT developer_id FROM activity_base_v2 WHERE developer_id IS NOT NULL
UNION
SELECT DISTINCT developer_id FROM contact_one_row_v2 WHERE developer_id IS NOT NULL
""")

date_summary = con.execute("""
SELECT MIN(activity_date) AS min_activity_date, MAX(activity_date) AS max_activity_date
FROM activity_ontology_v2
""").fetchdf()
ANCHOR_DATE = date_summary.loc[0, "max_activity_date"]
print("ANCHOR_DATE:", ANCHOR_DATE)

display(con.execute("""
SELECT
    COUNT(*) AS universe_developers,
    SUM(CASE WHEN a.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS developers_with_activity,
    SUM(CASE WHEN c.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS developers_with_contact,
    SUM(CASE WHEN a.developer_id IS NOT NULL AND c.developer_id IS NULL THEN 1 ELSE 0 END) AS activity_without_contact,
    SUM(CASE WHEN a.developer_id IS NULL AND c.developer_id IS NOT NULL THEN 1 ELSE 0 END) AS contact_without_activity
FROM developer_universe_v2 u
LEFT JOIN (SELECT DISTINCT developer_id FROM activity_base_v2) a USING (developer_id)
LEFT JOIN contact_one_row_v2 c USING (developer_id)
""").fetchdf())

ANCHOR_DATE: 2026-03-12 00:00:00


,universe_developers,developers_with_activity,developers_with_contact,activity_without_contact,contact_without_activity
0,9381508,7660278.0,9381490.0,18.0,1721230.0


## 7. Non-overlapping recency window features

The old notebook used cumulative windows. This version creates incremental windows to reduce collinearity:

- `0_30d`
- `30_90d`
- `90_180d`

In [61]:
WINDOWS = [
    ("0_30d", 0, 30),
    ("30_90d", 30, 90),
    ("90_180d", 90, 180),
]

def build_window_features(label: str, start_days_ago: int, end_days_ago: int) -> None:
    table_name = f"dev_features_{label}_v2"
    con.execute(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    WITH max_dt AS (SELECT MAX(activity_date) AS anchor_date FROM activity_ontology_v2),
    agg AS (
        SELECT
            developer_id,
            COUNT(*) AS activity_count,
            SUM(activity_score) AS activity_score_sum,
            AVG(activity_score) AS activity_score_avg,
            COUNT(DISTINCT activity_date) AS unique_activity_days,
            COUNT(DISTINCT activity) AS unique_activity_types,
            COUNT(DISTINCT modality) AS unique_modalities,
            COUNT(DISTINCT DATE_TRUNC('week', activity_date)) AS active_weeks,
            MIN(activity_date) AS first_activity_date_window,
            MAX(activity_date) AS last_activity_date_window,
            SUM(CASE WHEN journey_signal = 'Discover' THEN 1 ELSE 0 END) AS discover_count,
            SUM(CASE WHEN journey_signal = 'Learn' THEN 1 ELSE 0 END) AS learn_count,
            SUM(CASE WHEN journey_signal = 'Evaluate' THEN 1 ELSE 0 END) AS evaluate_count,
            SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS build_count,
            SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS champion_count,
            SUM(CASE WHEN effort_level = 'High' THEN 1 ELSE 0 END) AS high_effort_count,
            SUM(CASE WHEN modality = 'Download' THEN 1 ELSE 0 END) AS download_count,
            SUM(CASE WHEN modality = 'Hosted API' THEN 1 ELSE 0 END) AS hosted_api_count,
            SUM(CASE WHEN modality = 'Cloud Workspace' THEN 1 ELSE 0 END) AS cloud_workspace_count,
            SUM(CASE WHEN modality = 'Community' THEN 1 ELSE 0 END) AS community_count,
            SUM(CASE WHEN modality = 'Training' THEN 1 ELSE 0 END) AS training_count,
            SUM(CASE WHEN modality = 'Event' THEN 1 ELSE 0 END) AS event_count,
            SUM(cuda_persona_score) AS cuda_score,
            SUM(genai_persona_score) AS genai_score,
            SUM(robotics_persona_score) AS robotics_score,
            SUM(simulation_persona_score) AS simulation_score,
            SUM(learning_community_persona_score) AS learning_community_score
        FROM activity_ontology_v2, max_dt
        WHERE activity_date > anchor_date - INTERVAL {end_days_ago} DAY
          AND activity_date <= anchor_date - INTERVAL {start_days_ago} DAY
        GROUP BY developer_id
    )
    SELECT
        u.developer_id,
        COALESCE(a.activity_count, 0) AS activity_count,
        COALESCE(a.activity_score_sum, 0) AS activity_score_sum,
        COALESCE(a.activity_score_avg, 0) AS activity_score_avg,
        COALESCE(a.unique_activity_days, 0) AS unique_activity_days,
        COALESCE(a.unique_activity_types, 0) AS unique_activity_types,
        COALESCE(a.unique_modalities, 0) AS unique_modalities,
        COALESCE(a.active_weeks, 0) AS active_weeks,
        a.first_activity_date_window,
        a.last_activity_date_window,
        CASE WHEN a.last_activity_date_window IS NULL THEN 1 ELSE 0 END AS is_missing_last_activity_window,
        DATE_DIFF('day', a.last_activity_date_window, (SELECT anchor_date FROM max_dt)) AS days_since_last_activity_window,
        COALESCE(a.discover_count, 0) AS discover_count,
        COALESCE(a.learn_count, 0) AS learn_count,
        COALESCE(a.evaluate_count, 0) AS evaluate_count,
        COALESCE(a.build_count, 0) AS build_count,
        COALESCE(a.champion_count, 0) AS champion_count,
        COALESCE(a.high_effort_count, 0) AS high_effort_count,
        COALESCE(a.download_count, 0) AS download_count,
        COALESCE(a.hosted_api_count, 0) AS hosted_api_count,
        COALESCE(a.cloud_workspace_count, 0) AS cloud_workspace_count,
        COALESCE(a.community_count, 0) AS community_count,
        COALESCE(a.training_count, 0) AS training_count,
        COALESCE(a.event_count, 0) AS event_count,
        COALESCE(a.cuda_score, 0) AS cuda_score,
        COALESCE(a.genai_score, 0) AS genai_score,
        COALESCE(a.robotics_score, 0) AS robotics_score,
        COALESCE(a.simulation_score, 0) AS simulation_score,
        COALESCE(a.learning_community_score, 0) AS learning_community_score,
        CASE WHEN COALESCE(a.activity_count, 0) > 0 THEN 1 ELSE 0 END AS has_activity,
        LN(1 + COALESCE(a.activity_count, 0)) AS log_activity_count,
        LN(1 + COALESCE(a.activity_score_sum, 0)) AS log_activity_score_sum,
        LN(1 + COALESCE(a.build_count, 0)) AS log_build_count,
        LN(1 + COALESCE(a.high_effort_count, 0)) AS log_high_effort_count,
        COALESCE(a.activity_count, 0) * 1.0 / NULLIF(COALESCE(a.unique_activity_days, 0), 0) AS activity_per_active_day,
        COALESCE(a.build_count, 0) * 1.0 / NULLIF(COALESCE(a.activity_count, 0), 0) AS build_share,
        COALESCE(a.high_effort_count, 0) * 1.0 / NULLIF(COALESCE(a.activity_count, 0), 0) AS high_effort_share
    FROM developer_universe_v2 u
    LEFT JOIN agg a USING (developer_id)
    """)

for label, start, end in WINDOWS:
    build_window_features(label, start, end)
    print(f"Built dev_features_{label}_v2")
    display(con.execute(f"""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT developer_id) AS developers,
           AVG(has_activity) AS pct_with_activity,
           MAX(activity_count) AS max_activity_count
    FROM dev_features_{label}_v2
    """).fetchdf())

100% ▕██████████████████████████████████████▏ (00:00:04.11 elapsed)     
Built dev_features_0_30d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9381508,9381508,0.044561,144228


100% ▕██████████████████████████████████████▏ (00:00:04.08 elapsed)     
Built dev_features_30_90d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9381508,9381508,0.045331,272221


100% ▕██████████████████████████████████████▏ (00:00:04.21 elapsed)     
Built dev_features_90_180d_v2


,rows,developers,pct_with_activity,max_activity_count
0,9381508,9381508,0.055508,354569


## 8. Combine recency windows into one developer-level table

In [62]:
con.execute("""
CREATE OR REPLACE TABLE dev_recency_features_v2 AS
SELECT
    u.developer_id,

    f0.activity_count AS activity_count_0_30d,
    f1.activity_count AS activity_count_30_90d,
    f2.activity_count AS activity_count_90_180d,
    f0.has_activity AS has_activity_0_30d,
    f1.has_activity AS has_activity_30_90d,
    f2.has_activity AS has_activity_90_180d,
    f0.log_activity_count AS log_activity_count_0_30d,
    f1.log_activity_count AS log_activity_count_30_90d,
    f2.log_activity_count AS log_activity_count_90_180d,

    f0.build_count AS build_count_0_30d,
    f1.build_count AS build_count_30_90d,
    f2.build_count AS build_count_90_180d,
    f0.log_build_count AS log_build_count_0_30d,
    f1.log_build_count AS log_build_count_30_90d,
    f2.log_build_count AS log_build_count_90_180d,

    f0.high_effort_count AS high_effort_count_0_30d,
    f1.high_effort_count AS high_effort_count_30_90d,
    f2.high_effort_count AS high_effort_count_90_180d,

    f0.unique_activity_days AS unique_activity_days_0_30d,
    f0.unique_activity_types AS unique_activity_types_0_30d,
    f0.unique_modalities AS unique_modalities_0_30d,
    f0.activity_per_active_day AS activity_per_active_day_0_30d,
    f0.build_share AS build_share_0_30d,
    f0.high_effort_share AS high_effort_share_0_30d,

    f0.days_since_last_activity_window AS days_since_last_activity_0_30d,
    f0.is_missing_last_activity_window AS is_missing_last_activity_0_30d,

    -- Velocity and recency-decay features.
    f0.activity_count * 1.0 / NULLIF(f1.activity_count, 0) AS activity_velocity_0_30_vs_30_90,
    f0.build_count * 1.0 / NULLIF(f1.build_count, 0) AS build_velocity_0_30_vs_30_90,
    (0.60 * f0.activity_count + 0.30 * f1.activity_count + 0.10 * f2.activity_count) AS weighted_recent_activity,
    (0.60 * f0.build_count + 0.30 * f1.build_count + 0.10 * f2.build_count) AS weighted_recent_build,

    -- Interaction features.
    CASE WHEN f0.activity_count > 0 AND f0.build_count = 0 THEN 1 ELSE 0 END AS active_non_builder_0_30d,
    CASE WHEN f0.activity_count = 0 AND f1.activity_count > 0 THEN 1 ELSE 0 END AS newly_inactive_0_30d,
    CASE WHEN f0.build_count > 0 AND f0.activity_count <= 2 THEN 1 ELSE 0 END AS low_volume_builder_0_30d,
    CASE WHEN f0.high_effort_count > 0 THEN 1 ELSE 0 END AS has_high_effort_0_30d,
    CASE WHEN f0.build_count > 0 OR f0.hosted_api_count > 0 OR f0.cloud_workspace_count > 0 THEN 1 ELSE 0 END AS recent_build_flag,
    CASE WHEN f0.champion_count > 0 THEN 1 ELSE 0 END AS recent_champion_flag
FROM developer_universe_v2 u
LEFT JOIN dev_features_0_30d_v2 f0 USING (developer_id)
LEFT JOIN dev_features_30_90d_v2 f1 USING (developer_id)
LEFT JOIN dev_features_90_180d_v2 f2 USING (developer_id)
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    AVG(has_activity_0_30d) AS pct_active_0_30d,
    AVG(newly_inactive_0_30d) AS pct_newly_inactive_0_30d
FROM dev_recency_features_v2
""").fetchdf())

100% ▕██████████████████████████████████████▏ (00:00:04.40 elapsed)     


,rows,developers,pct_active_0_30d,pct_newly_inactive_0_30d
0,9381508,9381508,0.044561,0.038


## 9. Lifetime features with normalization, log transforms, and clipping helpers

In [63]:
con.execute("""
CREATE OR REPLACE TABLE dev_features_lifetime_v2 AS
WITH agg AS (
    SELECT
        developer_id,
        COUNT(*) AS lifetime_activity_count,
        SUM(activity_score) AS lifetime_activity_score_sum,
        AVG(activity_score) AS lifetime_activity_score_avg,
        COUNT(DISTINCT activity_date) AS lifetime_unique_activity_days,
        COUNT(DISTINCT activity) AS lifetime_unique_activity_types,
        COUNT(DISTINCT modality) AS lifetime_unique_modalities,
        COUNT(DISTINCT DATE_TRUNC('week', activity_date)) AS lifetime_active_weeks,
        MIN(activity_date) AS lifetime_first_activity_date,
        MAX(activity_date) AS lifetime_last_activity_date,
        SUM(CASE WHEN journey_signal = 'Discover' THEN 1 ELSE 0 END) AS lifetime_discover_count,
        SUM(CASE WHEN journey_signal = 'Learn' THEN 1 ELSE 0 END) AS lifetime_learn_count,
        SUM(CASE WHEN journey_signal = 'Evaluate' THEN 1 ELSE 0 END) AS lifetime_evaluate_count,
        SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS lifetime_build_count,
        SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS lifetime_champion_count,
        SUM(CASE WHEN effort_level = 'High' THEN 1 ELSE 0 END) AS lifetime_high_effort_count,
        SUM(cuda_persona_score) AS cuda_score,
        SUM(genai_persona_score) AS genai_score,
        SUM(robotics_persona_score) AS robotics_score,
        SUM(simulation_persona_score) AS simulation_score,
        SUM(learning_community_persona_score) AS learning_community_score,
        SUM(CASE WHEN persona_hint = 'Other' THEN COALESCE(NULLIF(activity_score, 0), 1) ELSE 0 END) AS other_persona_score
    FROM activity_ontology_v2
    GROUP BY developer_id
),
filled AS (
    SELECT
        u.developer_id,
        COALESCE(a.lifetime_activity_count, 0) AS lifetime_activity_count,
        COALESCE(a.lifetime_activity_score_sum, 0) AS lifetime_activity_score_sum,
        COALESCE(a.lifetime_activity_score_avg, 0) AS lifetime_activity_score_avg,
        COALESCE(a.lifetime_unique_activity_days, 0) AS lifetime_unique_activity_days,
        COALESCE(a.lifetime_unique_activity_types, 0) AS lifetime_unique_activity_types,
        COALESCE(a.lifetime_unique_modalities, 0) AS lifetime_unique_modalities,
        COALESCE(a.lifetime_active_weeks, 0) AS lifetime_active_weeks,
        a.lifetime_first_activity_date,
        a.lifetime_last_activity_date,
        COALESCE(a.lifetime_discover_count, 0) AS lifetime_discover_count,
        COALESCE(a.lifetime_learn_count, 0) AS lifetime_learn_count,
        COALESCE(a.lifetime_evaluate_count, 0) AS lifetime_evaluate_count,
        COALESCE(a.lifetime_build_count, 0) AS lifetime_build_count,
        COALESCE(a.lifetime_champion_count, 0) AS lifetime_champion_count,
        COALESCE(a.lifetime_high_effort_count, 0) AS lifetime_high_effort_count,
        COALESCE(a.cuda_score, 0) AS cuda_score,
        COALESCE(a.genai_score, 0) AS genai_score,
        COALESCE(a.robotics_score, 0) AS robotics_score,
        COALESCE(a.simulation_score, 0) AS simulation_score,
        COALESCE(a.learning_community_score, 0) AS learning_community_score,
        COALESCE(a.other_persona_score, 0) AS other_persona_score
    FROM developer_universe_v2 u
    LEFT JOIN agg a USING (developer_id)
),
p99 AS (
    SELECT
        APPROX_QUANTILE(lifetime_activity_count, 0.99) AS p99_activity_count,
        APPROX_QUANTILE(lifetime_build_count, 0.99) AS p99_build_count,
        APPROX_QUANTILE(lifetime_high_effort_count, 0.99) AS p99_high_effort_count
    FROM filled
)
SELECT
    f.*,
    CASE WHEN lifetime_activity_count > 0 THEN 1 ELSE 0 END AS has_lifetime_activity,
    LN(1 + lifetime_activity_count) AS log_lifetime_activity_count,
    LN(1 + lifetime_activity_score_sum) AS log_lifetime_activity_score_sum,
    LN(1 + lifetime_build_count) AS log_lifetime_build_count,
    LN(1 + lifetime_high_effort_count) AS log_lifetime_high_effort_count,
    LEAST(lifetime_activity_count, p99.p99_activity_count) AS clipped_lifetime_activity_count_p99,
    LEAST(lifetime_build_count, p99.p99_build_count) AS clipped_lifetime_build_count_p99,
    lifetime_activity_count * 1.0 / NULLIF(lifetime_active_weeks, 0) AS activity_per_active_week_lifetime,
    lifetime_build_count * 1.0 / NULLIF(lifetime_activity_count, 0) AS build_share_lifetime,
    lifetime_high_effort_count * 1.0 / NULLIF(lifetime_activity_count, 0) AS high_effort_share_lifetime
FROM filled f
CROSS JOIN p99
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    MAX(lifetime_activity_count) AS max_raw_activity_count,
    MAX(clipped_lifetime_activity_count_p99) AS max_clipped_activity_count
FROM dev_features_lifetime_v2
""").fetchdf())

100% ▕██████████████████████████████████████▏ (00:00:41.31 elapsed)     


,rows,developers,max_raw_activity_count,max_clipped_activity_count
0,9381508,9381508,2027375,115


## 10. Contact-profile persona hints, then final persona

Activity-based persona is still primary. Contact profile text is added only after developer-level aggregation so it enriches persona without changing activity row counts.

In [64]:
con.execute(r"""
CREATE OR REPLACE TABLE dev_contact_persona_v2 AS
WITH base AS (
    SELECT
        developer_id,
        LOWER(
            COALESCE(development_areas, '') || ' ' ||
            COALESCE(fields_of_interest, '') || ' ' ||
            COALESCE(industry_segment_vertical, '')
        ) AS profile_text
    FROM contact_one_row_v2
)
SELECT
    developer_id,
    CASE WHEN REGEXP_MATCHES(profile_text, 'cuda|gpu|accelerated|hpc|rapids|cudnn') THEN 1.0 ELSE 0.0 END AS cuda_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'genai|generative|llm|ai|machine learning|deep learning|inference') THEN 1.0 ELSE 0.0 END AS genai_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'robot|isaac|ros|jetson|autonomous|edge') THEN 1.0 ELSE 0.0 END AS robotics_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'simulation|omniverse|digital twin|graphics|render|rtx') THEN 1.0 ELSE 0.0 END AS simulation_profile_score,
    CASE WHEN REGEXP_MATCHES(profile_text, 'training|education|student|academic|community|developer program') THEN 1.0 ELSE 0.0 END AS learning_community_profile_score
FROM base
""")

con.execute("""
CREATE OR REPLACE TABLE dev_persona_v2 AS
WITH base AS (
    SELECT
        lf.developer_id,
        lf.cuda_score + COALESCE(cp.cuda_profile_score, 0) AS cuda_score,
        lf.genai_score + COALESCE(cp.genai_profile_score, 0) AS genai_score,
        lf.robotics_score + COALESCE(cp.robotics_profile_score, 0) AS robotics_score,
        lf.simulation_score + COALESCE(cp.simulation_profile_score, 0) AS simulation_score,
        lf.learning_community_score + COALESCE(cp.learning_community_profile_score, 0) AS learning_community_score,
        lf.other_persona_score
    FROM dev_features_lifetime_v2 lf
    LEFT JOIN dev_contact_persona_v2 cp USING (developer_id)
),
norm AS (
    SELECT *,
        cuda_score + genai_score + robotics_score + simulation_score + learning_community_score AS specific_persona_score,
        cuda_score + genai_score + robotics_score + simulation_score + learning_community_score + other_persona_score AS total_persona_score
    FROM base
),
shares AS (
    SELECT *,
        COALESCE(cuda_score / NULLIF(specific_persona_score, 0), 0) AS cuda_share,
        COALESCE(genai_score / NULLIF(specific_persona_score, 0), 0) AS genai_share,
        COALESCE(robotics_score / NULLIF(specific_persona_score, 0), 0) AS robotics_share,
        COALESCE(simulation_score / NULLIF(specific_persona_score, 0), 0) AS simulation_share,
        COALESCE(learning_community_score / NULLIF(specific_persona_score, 0), 0) AS learning_community_share,
        COALESCE(other_persona_score / NULLIF(total_persona_score, 0), 0) AS other_share
    FROM norm
),
entropy AS (
    SELECT *,
        -1 * (
            CASE WHEN cuda_share > 0 THEN cuda_share * LN(cuda_share) ELSE 0 END +
            CASE WHEN genai_share > 0 THEN genai_share * LN(genai_share) ELSE 0 END +
            CASE WHEN robotics_share > 0 THEN robotics_share * LN(robotics_share) ELSE 0 END +
            CASE WHEN simulation_share > 0 THEN simulation_share * LN(simulation_share) ELSE 0 END +
            CASE WHEN learning_community_share > 0 THEN learning_community_share * LN(learning_community_share) ELSE 0 END
        ) / LN(5) AS persona_entropy
    FROM shares
),
long_scores AS (
    SELECT developer_id, 'CUDA' AS persona, cuda_share AS score FROM entropy
    UNION ALL SELECT developer_id, 'GenAI', genai_share FROM entropy
    UNION ALL SELECT developer_id, 'Robotics', robotics_share FROM entropy
    UNION ALL SELECT developer_id, 'Simulation', simulation_share FROM entropy
    UNION ALL SELECT developer_id, 'Learning_Community', learning_community_share FROM entropy
),
ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY developer_id ORDER BY score DESC, persona) AS rn,
        LEAD(score) OVER (PARTITION BY developer_id ORDER BY score DESC, persona) AS second_score
    FROM long_scores
)
SELECT
    e.*,
    CASE WHEN e.specific_persona_score = 0 THEN 'Unknown' ELSE r.persona END AS persona,
    CASE WHEN e.specific_persona_score = 0 THEN 0 ELSE r.score END AS persona_confidence,
    CASE
        WHEN e.specific_persona_score = 0 THEN 'Unknown'
        WHEN r.score >= 0.70 THEN 'High'
        WHEN r.score >= 0.45 THEN 'Medium'
        ELSE 'Low'
    END AS persona_confidence_tier,
    CASE
        WHEN e.specific_persona_score = 0 THEN 0
        WHEN e.persona_entropy >= 0.60 OR r.score - COALESCE(r.second_score, 0) <= 0.15 THEN 1
        ELSE 0
    END AS mixed_persona_flag
FROM entropy e
LEFT JOIN ranked r ON e.developer_id = r.developer_id AND r.rn = 1
""")

display(con.execute("""
SELECT persona, persona_confidence_tier, mixed_persona_flag, COUNT(*) AS developers
FROM dev_persona_v2
GROUP BY 1,2,3
ORDER BY developers DESC
""").fetchdf())

100% ▕██████████████████████████████████████▏ (00:00:03.13 elapsed)     


,persona,persona_confidence_tier,mixed_persona_flag,developers
0,CUDA,High,0,1899533
1,Unknown,Unknown,0,1875549
2,GenAI,High,0,1235727
3,CUDA,Medium,0,795937
4,GenAI,Medium,1,607852
5,CUDA,Medium,1,564741
6,Simulation,High,0,455345
7,GenAI,Low,1,321074
8,Robotics,High,0,308007
9,Learning_Community,High,0,260607


## 11. Activation and dormancy status

This block now defines **Unactivated** from lifetime activity, not from meaningful-week rules. This avoids incorrectly labeling users with historical activity as unactivated. Dormancy uses last activity recency with broader thresholds so dormant does not dominate the journey labels.

Dormancy logic:

- `Unactivated`: lifetime activity count is zero
- `Active`: last activity within 30 days
- `Cooling`: last activity 30-90 days ago
- `At_Risk`: last activity 90-365 days ago
- `Dormant`: last activity 365+ days ago

Meaningful weeks are still kept as supporting context, but they no longer determine whether a developer has ever activated.

In [65]:
ACTIVE_CUTOFF_DAYS = 30
COOLING_CUTOFF_DAYS = 90
DORMANT_CUTOFF_DAYS = 365

con.execute("""
CREATE OR REPLACE TABLE dev_weekly_features_v2 AS
SELECT
    developer_id,
    DATE_TRUNC('week', activity_date) AS week_start,
    COUNT(*) AS activity_count_total,
    SUM(activity_score) AS activity_score_sum,
    COUNT(DISTINCT activity) AS unique_activity_types,
    SUM(CASE WHEN journey_signal = 'Build' THEN 1 ELSE 0 END) AS build_count,
    SUM(CASE WHEN journey_signal = 'Champion' THEN 1 ELSE 0 END) AS champion_count,
    SUM(CASE WHEN effort_level = 'High' THEN 1 ELSE 0 END) AS high_effort_count,
    SUM(CASE WHEN modality IN ('Hosted API', 'Cloud Workspace') THEN 1 ELSE 0 END) AS product_use_count
FROM activity_ontology_v2
GROUP BY 1,2
""")

con.execute("""
CREATE OR REPLACE TABLE dev_meaningful_week_v2 AS
SELECT *,
    CASE
        WHEN build_count > 0
          OR champion_count > 0
          OR high_effort_count > 0
          OR product_use_count > 0
          OR activity_count_total >= 3
        THEN 1 ELSE 0
    END AS meaningful_week_flag
FROM dev_weekly_features_v2
""")

con.execute("""
CREATE OR REPLACE TABLE dev_activation_v2 AS
WITH meaningful AS (
    SELECT
        developer_id,
        COALESCE(SUM(meaningful_week_flag), 0) AS lifetime_meaningful_weeks,
        MIN(CASE WHEN meaningful_week_flag = 1 THEN week_start END) AS first_meaningful_week_start,
        MAX(CASE WHEN meaningful_week_flag = 1 THEN week_start END) AS last_meaningful_week_start
    FROM dev_meaningful_week_v2
    GROUP BY developer_id
)
SELECT
    u.developer_id,
    CASE WHEN COALESCE(l.lifetime_activity_count, 0) > 0 THEN 1 ELSE 0 END AS is_activated,
    COALESCE(l.lifetime_activity_count, 0) AS lifetime_activity_count_for_activation,
    COALESCE(m.lifetime_meaningful_weeks, 0) AS lifetime_meaningful_weeks,
    m.first_meaningful_week_start,
    m.last_meaningful_week_start,
    l.lifetime_first_activity_date,
    l.lifetime_last_activity_date
FROM developer_universe_v2 u
LEFT JOIN dev_features_lifetime_v2 l USING (developer_id)
LEFT JOIN meaningful m USING (developer_id)
""")

con.execute(f"""
CREATE OR REPLACE TABLE dev_dormancy_status_v2 AS
WITH max_dt AS (
    SELECT MAX(activity_date) AS anchor_date FROM activity_ontology_v2
),
base AS (
    SELECT
        a.*,
        CASE
            WHEN a.lifetime_last_activity_date IS NOT NULL
            THEN DATE_DIFF('day', a.lifetime_last_activity_date, (SELECT anchor_date FROM max_dt))
            ELSE NULL
        END AS days_since_last_activity,
        CASE
            WHEN a.last_meaningful_week_start IS NOT NULL
            THEN DATE_DIFF('day', a.last_meaningful_week_start, (SELECT anchor_date FROM max_dt))
            ELSE NULL
        END AS days_since_last_meaningful_week
    FROM dev_activation_v2 a
)
SELECT
    *,
    CASE
        WHEN lifetime_activity_count_for_activation = 0 THEN 'Unactivated'
        WHEN days_since_last_activity < {ACTIVE_CUTOFF_DAYS} THEN 'Active'
        WHEN days_since_last_activity < {COOLING_CUTOFF_DAYS} THEN 'Cooling'
        WHEN days_since_last_activity < {DORMANT_CUTOFF_DAYS} THEN 'At_Risk'
        ELSE 'Dormant'
    END AS dormancy_status,
    CASE
        WHEN lifetime_activity_count_for_activation > 0
         AND days_since_last_activity >= {DORMANT_CUTOFF_DAYS}
        THEN 1 ELSE 0
    END AS dormant_flag,
    CASE
        WHEN lifetime_activity_count_for_activation > 0
         AND days_since_last_activity >= {COOLING_CUTOFF_DAYS}
         AND days_since_last_activity < {DORMANT_CUTOFF_DAYS}
        THEN 1 ELSE 0
    END AS at_risk_flag,
    CASE
        WHEN lifetime_activity_count_for_activation > 0
         AND days_since_last_activity >= {ACTIVE_CUTOFF_DAYS}
         AND days_since_last_activity < {COOLING_CUTOFF_DAYS}
        THEN 1 ELSE 0
    END AS cooling_flag
FROM base
""")

display(con.execute("""
SELECT
    dormancy_status,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct,
    AVG(days_since_last_activity) AS avg_days_since_last_activity,
    MEDIAN(days_since_last_activity) AS median_days_since_last_activity,
    AVG(days_since_last_meaningful_week) AS avg_days_since_last_meaningful_week
FROM dev_dormancy_status_v2
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())


100% ▕██████████████████████████████████████▏ (00:00:08.10 elapsed)     


,dormancy_status,developers,pct,avg_days_since_last_activity,median_days_since_last_activity,avg_days_since_last_meaningful_week
0,Dormant,5304852,56.55,1103.341224,1037.0,1206.891733
1,Unactivated,1721230,18.35,NaN,NaN,NaN
2,At_Risk,1580877,16.85,237.897645,240.0,278.209169
3,Active,418049,4.46,11.338460,10.0,35.046828
4,Cooling,356500,3.80,55.597027,54.0,104.299126


## 12. Journey state features from the recent window

Journey state is now built **after** dormancy. It uses lifetime activity to define `Unactivated`, then applies dormancy states, then uses recent intent and depth. Effort is an input, not a one-to-one mapping to journey stage.

In [67]:
con.execute("""
CREATE OR REPLACE TABLE dev_journey_state_v2 AS
WITH base AS (
    SELECT
        r.developer_id,

        -- Recent activity
        COALESCE(r.activity_count_0_30d, 0) AS activity_count_0_30d,
        COALESCE(r.activity_count_30_90d, 0) AS activity_count_30_90d,
        COALESCE(r.activity_count_90_180d, 0) AS activity_count_90_180d,

        COALESCE(r.build_count_0_30d, 0) AS build_count_0_30d,
        COALESCE(r.build_count_30_90d, 0) AS build_count_30_90d,

        COALESCE(r.high_effort_count_0_30d, 0) AS high_effort_count_0_30d,
        COALESCE(r.high_effort_count_30_90d, 0) AS high_effort_count_30_90d,

        COALESCE(r.unique_activity_types_0_30d, 0) AS unique_activity_types_0_30d,
        COALESCE(r.unique_modalities_0_30d, 0) AS unique_modalities_0_30d,

        COALESCE(r.recent_build_flag, 0) AS recent_build_flag,
        COALESCE(r.recent_champion_flag, 0) AS recent_champion_flag,

        -- Dormancy / activation
        COALESCE(d.is_activated, 0) AS is_activated,
        COALESCE(d.lifetime_activity_count_for_activation, 0) AS lifetime_activity_count_for_activation,
        COALESCE(d.dormancy_status, 'Unactivated') AS dormancy_status,
        COALESCE(d.dormant_flag, 0) AS dormant_flag,
        COALESCE(d.at_risk_flag, 0) AS at_risk_flag,
        COALESCE(d.cooling_flag, 0) AS cooling_flag,
        d.days_since_last_activity,

        -- Lifetime context
        COALESCE(l.lifetime_activity_count, 0) AS lifetime_activity_count,
        COALESCE(l.lifetime_build_count, 0) AS lifetime_build_count,
        COALESCE(l.lifetime_champion_count, 0) AS lifetime_champion_count,

        -- Trend
        CASE
            WHEN COALESCE(r.activity_count_30_90d, 0) = 0
             AND COALESCE(r.activity_count_0_30d, 0) > 0 THEN 2.0
            WHEN COALESCE(r.activity_count_30_90d, 0) = 0 THEN 0.0
            ELSE CAST(r.activity_count_0_30d AS DOUBLE)
                 / NULLIF(CAST(r.activity_count_30_90d AS DOUBLE), 0)
        END AS recent_activity_trend_ratio

    FROM dev_recency_features_v2 r
    LEFT JOIN dev_dormancy_status_v2 d
        ON r.developer_id = d.developer_id
    LEFT JOIN dev_features_lifetime_v2 l
        ON r.developer_id = l.developer_id
),

scored AS (
    SELECT
        *,

        CASE
            WHEN activity_count_0_30d >= 5 THEN 'High'
            WHEN activity_count_0_30d >= 2 THEN 'Medium'
            WHEN activity_count_0_30d = 1 THEN 'Low'
            ELSE 'None'
        END AS activity_volume_band,

        CASE
            WHEN build_count_0_30d > 0 OR recent_build_flag = 1 THEN 'Build_Intent'
            WHEN high_effort_count_0_30d > 0 THEN 'Evaluation_Intent'
            WHEN activity_count_0_30d > 0 THEN 'Learning_Intent'
            ELSE 'No_Recent_Intent'
        END AS intent_signal,

        CASE
            WHEN activity_count_0_30d = 0 THEN 'Inactive'
            WHEN recent_activity_trend_ratio >= 1.5 THEN 'Accelerating'
            WHEN recent_activity_trend_ratio >= 0.7 THEN 'Stable'
            WHEN recent_activity_trend_ratio > 0 THEN 'Declining'
            ELSE 'New_or_Returned'
        END AS trend_signal

    FROM base
)

SELECT
    developer_id,

    CASE
        -- True never-active users only
        WHEN lifetime_activity_count = 0 THEN 'Unactivated'

        -- Dormancy should override effort, but only after true activation is known
        WHEN dormancy_status = 'Dormant' THEN 'Dormant'
        WHEN dormancy_status = 'At_Risk' THEN 'At_Risk'
        WHEN dormancy_status = 'Cooling' AND activity_count_0_30d = 0 THEN 'Cooling'

        -- Highest-intent active users
        WHEN recent_champion_flag = 1
          OR (lifetime_champion_count > 0 AND activity_count_0_30d > 0)
        THEN 'Champion'

        -- Active builders
        WHEN build_count_0_30d > 0
          OR recent_build_flag = 1
        THEN 'Builder'

        -- High-intent evaluation, but not yet building
        WHEN high_effort_count_0_30d > 0
          AND build_count_0_30d = 0
        THEN 'Evaluator'

        -- Active but mostly low-depth usage
        WHEN activity_count_0_30d > 0
          AND unique_activity_types_0_30d >= 2
        THEN 'Explorer'

        -- Minimal recent activity
        WHEN activity_count_0_30d > 0
        THEN 'Learner'

        -- Previously active, not recently active, but not yet dormant enough
        ELSE 'Inactive'
    END AS current_journey_state_30d,

    CASE
        WHEN lifetime_activity_count = 0 THEN 0
        WHEN dormancy_status = 'Dormant' THEN 1
        WHEN dormancy_status = 'At_Risk' THEN 2
        WHEN dormancy_status = 'Cooling' AND activity_count_0_30d = 0 THEN 3
        WHEN activity_count_0_30d = 0 THEN 3
        WHEN activity_count_0_30d > 0 AND build_count_0_30d = 0 AND high_effort_count_0_30d = 0 THEN 4
        WHEN high_effort_count_0_30d > 0 AND build_count_0_30d = 0 THEN 5
        WHEN build_count_0_30d > 0 OR recent_build_flag = 1 THEN 6
        WHEN recent_champion_flag = 1 OR lifetime_champion_count > 0 THEN 7
        ELSE 3
    END AS current_journey_rank_30d,

    activity_volume_band,
    intent_signal,
    trend_signal,
    recent_activity_trend_ratio

FROM scored
""")

display(con.execute("""
SELECT
    current_journey_state_30d,
    COUNT(*) AS developers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM dev_journey_state_v2
GROUP BY 1
ORDER BY developers DESC
""").fetchdf())

# Quick guardrail: Unactivated should mean no lifetime activity
display(con.execute("""
SELECT COUNT(*) AS unactivated_with_lifetime_activity
FROM dev_journey_state_v2 j
JOIN dev_features_lifetime_v2 l USING (developer_id)
WHERE j.current_journey_state_30d = 'Unactivated'
  AND l.lifetime_activity_count > 0
 """).fetchdf())


,current_journey_state_30d,developers,pct
0,Dormant,5304852,56.55
1,Unactivated,1721230,18.35
2,At_Risk,1580877,16.85
3,Cooling,356500,3.80
4,Learner,206595,2.20
5,Builder,182281,1.94
6,Explorer,19117,0.20
7,Champion,10056,0.11


,unactivated_with_lifetime_activity
0,0


## 13. Final profile: join contact metadata last

In [68]:
con.execute("""
CREATE OR REPLACE TABLE dev_profile_final_v4 AS
SELECT
    u.developer_id,

    -- Persona
    p.persona,
    p.persona_confidence,
    p.persona_confidence_tier,
    p.persona_entropy,
    p.mixed_persona_flag,
    p.cuda_share,
    p.genai_share,
    p.robotics_share,
    p.simulation_share,
    p.learning_community_share,

    -- Journey and dormancy
    js.current_journey_state_30d,
    js.current_journey_rank_30d,
    d.is_activated,
    d.lifetime_meaningful_weeks,
    d.last_meaningful_week_start,
    d.days_since_last_activity,
    d.days_since_last_meaningful_week,
    d.dormancy_status,
    d.dormant_flag,
    d.at_risk_flag,
    d.cooling_flag,

    -- Recency / incremental windows
    r.* EXCLUDE (developer_id),

    -- Lifetime features
    lf.* EXCLUDE (developer_id, cuda_score, genai_score, robotics_score, simulation_score, learning_community_score, other_persona_score),

    -- Contact enrichment, intentionally last
    c.created_date AS contact_created_date,
    c.first_activity_date AS contact_first_activity_date,
    c.last_activity_date AS contact_last_activity_date,
    c.account_id,
    c.account_type,
    c.country,
    c.region,
    c.industry_segment_vertical,
    c.program_application_source,
    c.organization_english_name,
    c.normalized_account_name,
    c.wwfo_category,
    c.wwfo_target_list,
    CASE WHEN c.developer_id IS NULL THEN 1 ELSE 0 END AS missing_contact_metadata_flag
FROM developer_universe_v2 u
LEFT JOIN dev_persona_v2 p USING (developer_id)
LEFT JOIN dev_journey_state_v2 js USING (developer_id)
LEFT JOIN dev_dormancy_status_v2 d USING (developer_id)
LEFT JOIN dev_recency_features_v2 r USING (developer_id)
LEFT JOIN dev_features_lifetime_v2 lf USING (developer_id)
LEFT JOIN contact_one_row_v2 c USING (developer_id)
""")

display(con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS developers,
    SUM(CASE WHEN persona IS NULL THEN 1 ELSE 0 END) AS missing_persona,
    SUM(CASE WHEN current_journey_state_30d IS NULL THEN 1 ELSE 0 END) AS missing_journey_state,
    SUM(missing_contact_metadata_flag) AS missing_contact_metadata
FROM dev_profile_final_v4
""").fetchdf())

display(con.execute("""
SELECT persona, current_journey_state_30d, dormancy_status, COUNT(*) AS developers
FROM dev_profile_final_v4
GROUP BY 1,2,3
ORDER BY developers DESC
LIMIT 40
""").fetchdf())

100% ▕██████████████████████████████████████▏ (00:00:15.57 elapsed)     


,rows,developers,missing_persona,missing_journey_state,missing_contact_metadata
0,9381508,9381508,0.0,0.0,18.0


,persona,current_journey_state_30d,dormancy_status,developers
0,CUDA,Dormant,Dormant,2671164
1,Unknown,Unactivated,Unactivated,1293261
2,GenAI,Dormant,Dormant,1089570
3,CUDA,At_Risk,At_Risk,616444
4,GenAI,At_Risk,At_Risk,538608
5,Unknown,Dormant,Dormant,419763
6,Learning_Community,Dormant,Dormant,409125
7,Simulation,Dormant,Dormant,380966
8,Robotics,Dormant,Dormant,334264
9,GenAI,Unactivated,Unactivated,283810


## 14. Feature validation checklist

In [69]:
validation = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT developer_id) AS distinct_developers,
    SUM(CASE WHEN activity_count_0_30d < 0 OR lifetime_activity_count < 0 THEN 1 ELSE 0 END) AS negative_count_violations,
    SUM(CASE WHEN persona_confidence < 0 OR persona_confidence > 1 THEN 1 ELSE 0 END) AS persona_confidence_violations,
    SUM(CASE WHEN persona_entropy < 0 OR persona_entropy > 1 THEN 1 ELSE 0 END) AS persona_entropy_violations,
    SUM(CASE WHEN has_activity_0_30d = 0 AND activity_count_0_30d <> 0 THEN 1 ELSE 0 END) AS zero_flag_violations,
    SUM(CASE WHEN recent_build_flag = 1 AND build_count_0_30d = 0 THEN 1 ELSE 0 END) AS recent_build_flag_violations,
    SUM(CASE WHEN missing_contact_metadata_flag NOT IN (0,1) THEN 1 ELSE 0 END) AS contact_flag_violations,
    SUM(CASE WHEN current_journey_state_30d = 'Unactivated' AND lifetime_activity_count > 0 THEN 1 ELSE 0 END) AS unactivated_with_lifetime_activity_violations
FROM dev_profile_final_v4
""").fetchdf()

display(validation)

if validation.loc[0, "rows"] != validation.loc[0, "distinct_developers"]:
    raise ValueError("Final table is not one row per developer")

violation_cols = [c for c in validation.columns if c.endswith("violations")]
violations = validation.loc[0, violation_cols].sum()
if violations > 0:
    print("WARNING: validation violations found. Inspect the table above before modeling.")
else:
    print("All feature validation checks passed.")

,rows,distinct_developers,negative_count_violations,persona_confidence_violations,persona_entropy_violations,zero_flag_violations,recent_build_flag_violations,contact_flag_violations,unactivated_with_lifetime_activity_violations
0,9381508,9381508,0.0,0.0,1.0,0.0,17.0,0.0,0.0


## 15. Table inventory

In [70]:
final_tables = [
    "activity_base_v2",
    "contact_one_row_v2",
    "activity_ontology_v2",
    "developer_universe_v2",
    "dev_features_0_30d_v2",
    "dev_features_30_90d_v2",
    "dev_features_90_180d_v2",
    "dev_recency_features_v2",
    "dev_features_lifetime_v2",
    "dev_contact_persona_v2",
    "dev_persona_v2",
    "dev_journey_state_v2",
    "dev_weekly_features_v2",
    "dev_meaningful_week_v2",
    "dev_activation_v2",
    "dev_dormancy_status_v2",
    "dev_profile_final_v4",
]

inventory = []
for t in final_tables:
    exists = con.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = ?", [t]).fetchone()[0] > 0
    rows = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] if exists else None
    inventory.append({"table": t, "rows": rows})

display(pd.DataFrame(inventory))

,table,rows
0,activity_base_v2,69347501
1,contact_one_row_v2,9381490
2,activity_ontology_v2,69347501
3,developer_universe_v2,9381508
4,dev_features_0_30d_v2,9381508
5,dev_features_30_90d_v2,9381508
6,dev_features_90_180d_v2,9381508
7,dev_recency_features_v2,9381508
8,dev_features_lifetime_v2,9381508
9,dev_contact_persona_v2,9381490


In [72]:
# Close when finished.
con.close()